# Chapter 15: Probability Models in Structural Engineering
## The Normal Distribution, Load Factors, and Why Safety Factors Are Not Guesses

**Learning Objectives:**
- Describe the normal distribution and identify its parameters (mean, standard deviation)
- Explain how structural loads and material resistances both follow normal distributions
- Calculate the probability of structural failure as the overlap between load and resistance curves
- Connect the reliability index β to everyday probability language


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(15)
print('Setup complete. (scipy is pre-installed in Colab)')


## 15.1  The Normal Distribution in Structural Engineering

Two of the most important quantities in structural engineering — **loads** and **resistances** — both follow the **normal (bell-curve) distribution**:

| Quantity | What It Represents | Typical Mean | Typical CV* |
|----------|--------------------|-------------|------------|
| **Load S** | Actual load applied to a member (dead + live) | Code-specified value | 10–30% |
| **Resistance R** | Actual strength of a member (steel yield × section) | Above nominal | 10–15% |

*CV = coefficient of variation = σ/μ, a measure of relative scatter.

A structure **fails** when the actual load exceeds the actual resistance: **S > R**, or equivalently **R − S < 0**.

Because both R and S are random variables, the difference **Z = R − S** is also normally distributed:

$$Z = R - S \sim N(\mu_R - \mu_S,\; \sqrt{\sigma_R^2 + \sigma_S^2})$$

The **probability of failure** is the probability that Z < 0 — which is the area of the overlapping tails of the two distributions.

### The Reliability Index β

Engineers compress all of this into a single number called the **reliability index**:

$$\beta = \frac{\mu_R - \mu_S}{\sqrt{\sigma_R^2 + \sigma_S^2}}$$

β measures how many standard deviations separate the mean resistance from the mean load. Modern codes (LRFD) are calibrated so that typical structural members achieve **β ≈ 3.5**, corresponding to a failure probability of about **1 in 5,000** per year.


## 🔬 Interactive Experiment 1: Load vs. Resistance — Finding the Overlap

The widget below shows two overlapping normal distributions:
- **Red curve**: the distribution of actual loads S applied to a beam connection
- **Blue curve**: the distribution of actual resistance R (connection strength)

The **shaded overlap region** represents combinations where S > R — structural failure. Adjust the means and spreads and watch how the failure probability changes.


In [ ]:
def _lrfd_plot(mu_R, sigma_R, mu_S, sigma_S):
    x = np.linspace(0, 300, 1000)
    pdf_R = stats.norm.pdf(x, mu_R, sigma_R)
    pdf_S = stats.norm.pdf(x, mu_S, sigma_S)

    beta = (mu_R - mu_S) / np.sqrt(sigma_R**2 + sigma_S**2)
    p_fail = stats.norm.cdf(-beta)  # P(Z < 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(x, pdf_R, 'steelblue', lw=2.5, label=f'Resistance R  (μ={mu_R}, σ={sigma_R})')
    ax1.plot(x, pdf_S, 'firebrick',  lw=2.5, label=f'Load S  (μ={mu_S}, σ={sigma_S})')
    overlap_x = x[(x >= mu_S - 3*sigma_S) & (x <= mu_R + 3*sigma_R)]
    overlap_y = np.minimum(
        stats.norm.pdf(overlap_x, mu_R, sigma_R),
        stats.norm.pdf(overlap_x, mu_S, sigma_S)
    )
    ax1.fill_between(overlap_x, overlap_y, alpha=0.35, color='purple', label='Failure zone (S > R)')
    ax1.axvline(mu_R, color='steelblue', ls='--', lw=1.5)
    ax1.axvline(mu_S, color='firebrick',  ls='--', lw=1.5)
    ax1.set_xlabel('Force (kips)', fontsize=12)
    ax1.set_ylabel('Probability Density', fontsize=12)
    ax1.set_title('Load and Resistance Distributions', fontsize=13)
    ax1.legend(fontsize=10)

    # Right: Z = R - S distribution
    mu_Z  = mu_R - mu_S
    sig_Z = np.sqrt(sigma_R**2 + sigma_S**2)
    xz = np.linspace(mu_Z - 4*sig_Z, mu_Z + 4*sig_Z, 500)
    pz = stats.norm.pdf(xz, mu_Z, sig_Z)
    ax2.plot(xz, pz, 'purple', lw=2.5, label='Z = R − S')
    fail_x = xz[xz <= 0]
    ax2.fill_between(fail_x, stats.norm.pdf(fail_x, mu_Z, sig_Z),
        alpha=0.4, color='firebrick', label=f'P(failure) = {p_fail:.2e}')
    ax2.axvline(0, color='black', lw=2, ls='--', label='Failure threshold (Z=0)')
    ax2.set_xlabel('Z = R − S (kips)', fontsize=12)
    ax2.set_ylabel('Probability Density', fontsize=12)
    ax2.set_title(f'Reliability Index  β = {beta:.2f}  |  P(fail) = {p_fail:.2e}', fontsize=13)
    ax2.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'β = {beta:.3f}   |   P(failure) ≈ {p_fail:.2e}   ({p_fail*100:.4f}%)')
    if beta >= 3.5:
        print('✅ Meets LRFD target (β ≥ 3.5)')
    else:
        print('❌ Below LRFD target (β < 3.5) — redesign needed')

w_muR  = widgets.IntSlider(value=150, min=80, max=250, step=5,
    description='Mean Resistance μ_R (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_sigR = widgets.IntSlider(value=15, min=5, max=40, step=1,
    description='SD Resistance σ_R (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_muS  = widgets.IntSlider(value=80, min=40, max=200, step=5,
    description='Mean Load μ_S (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_sigS = widgets.IntSlider(value=20, min=5, max=50, step=1,
    description='SD Load σ_S (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
out1 = widgets.interactive_output(_lrfd_plot,
    {'mu_R':w_muR,'sigma_R':w_sigR,'mu_S':w_muS,'sigma_S':w_sigS})
display(widgets.VBox([w_muR, w_sigR, w_muS, w_sigS, out1]))


---
## ⚠️  Why Engineers Moved From ASD to LRFD

For most of the 20th century, structural engineers used **Allowable Stress Design (ASD)**: divide the material's nominal strength by a single **factor of safety** (typically 1.5 to 2.5) to get an allowable stress, and make sure the calculated stress stays below it.

This was simple, but it had a hidden flaw: **it treated all uncertainty the same way.** Dead loads are known quite precisely (±10–15%). Live loads are much more uncertain (±25–30%). Wind and earthquake loads are highly uncertain (±40–60%). Using one safety factor for all of them meant the actual reliability was inconsistent — a steel beam in a storage warehouse had a very different true failure probability than a steel beam in a hospital lobby, even if both 'passed' ASD.

**Load and Resistance Factor Design (LRFD)** — now required by AISC for steel, ACI for concrete, and referenced throughout Hibbeler — uses *separate* factors for each load type:

$$1.2D + 1.6L \leq \phi R_n$$

Where:
- **1.2** and **1.6** are *load factors* — higher for more uncertain loads
- **φ** is the *resistance factor* (typically 0.9 for bending, 0.75 for shear) — accounting for material variability
- **R_n** is the *nominal resistance* computed from design equations

These numbers were not chosen by guessing. They were **calibrated using the reliability index β** so that typical structural members achieve a consistent probability of failure regardless of the load type or material.

> *The load factors and resistance factors in Hibbeler's design examples are the normal distribution's influence on structural engineering — translated into numbers a practicing engineer can use every day.*


## 🔬 Interactive Experiment 2: How β Changes With Load Uncertainty

A structural engineer is sizing a beam. The resistance (beam strength) is fixed. The load has a fixed mean, but the **uncertainty (standard deviation)** can vary depending on the load type: dead loads are known precisely, live loads less so, wind loads even less.

Watch how the reliability index β responds to changes in load uncertainty — and what that implies for how large the safety margin needs to be.


In [ ]:
def _beta_sensitivity(mu_R, sigma_R_pct, mu_S, sigma_S_pct):
    sigma_R = mu_R * sigma_R_pct / 100
    sigma_S_vals = np.linspace(1, 60, 200)
    betas = [(mu_R - mu_S) / np.sqrt(sigma_R**2 + (mu_S * sv/100)**2)
             for sv in sigma_S_vals]
    sigma_S_current = mu_S * sigma_S_pct / 100
    beta_current = (mu_R - mu_S) / np.sqrt(sigma_R**2 + sigma_S_current**2)
    p_fail = stats.norm.cdf(-beta_current)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(sigma_S_vals, betas, 'steelblue', lw=2.5)
    ax.axhline(3.5, color='green', ls='--', lw=2, label='LRFD target β = 3.5')
    ax.axhline(2.0, color='orange',ls='--', lw=1.5, label='Minimum acceptable β = 2.0')
    ax.axvline(sigma_S_pct, color='firebrick', ls='-', lw=2,
        label=f'Current σ_S = {sigma_S_pct:.0f}% → β = {beta_current:.2f}')
    ax.fill_betweenx([0,7], 0, sigma_S_vals[np.array(betas)>=3.5].max() if any(b>=3.5 for b in betas) else 0,
        alpha=0.08, color='green', label='β ≥ 3.5 zone')

    load_types = {'Dead load': 10, 'Live load': 25, 'Wind load': 40, 'Earthquake': 55}
    for lt, sv in load_types.items():
        ax.annotate(lt, xy=(sv, 0.3), fontsize=9, ha='center', color='gray')
        ax.axvline(sv, color='gray', ls=':', lw=1, alpha=0.5)

    ax.set_xlabel('Load Coefficient of Variation σ_S/μ_S (%)', fontsize=12)
    ax.set_ylabel('Reliability Index β', fontsize=12)
    ax.set_title('How Load Uncertainty Drives Required Safety Margin', fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 7)
    plt.tight_layout()
    plt.show()
    print(f'β = {beta_current:.2f}  |  P(failure) ≈ {p_fail:.2e}')

w_muR  = widgets.IntSlider(value=150, min=80, max=250, step=5,
    description='Mean Resistance μ_R (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_cvR  = widgets.IntSlider(value=10, min=3, max=20, step=1,
    description='Resistance CV % (σ_R/μ_R):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_muS  = widgets.IntSlider(value=80, min=40, max=200, step=5,
    description='Mean Load μ_S (kips):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
w_cvS  = widgets.IntSlider(value=25, min=1, max=60, step=1,
    description='Load CV % (σ_S/μ_S):', style={'description_width':'initial'},
    layout=widgets.Layout(width='440px'))
out2 = widgets.interactive_output(_beta_sensitivity,
    {'mu_R':w_muR,'sigma_R_pct':w_cvR,'mu_S':w_muS,'sigma_S_pct':w_cvS})
display(widgets.VBox([w_muR, w_cvR, w_muS, w_cvS, out2]))


---
## 📋  Chapter 15 Review

| Term | Meaning |
|------|--------|
| **Normal distribution** | Bell-shaped distribution described by mean μ and standard deviation σ |
| **Load S** | Random variable representing actual applied load — follows N(μ_S, σ_S) |
| **Resistance R** | Random variable representing actual structural strength — follows N(μ_R, σ_R) |
| **Reliability index β** | Number of standard deviations between mean resistance and mean load |
| **P(failure)** | Area where load distribution exceeds resistance distribution; P(Z<0) |
| **LRFD** | Load and Resistance Factor Design — code format calibrated to achieve β ≈ 3.5 |

**The Big Idea:** Every load factor (1.2, 1.6) and every resistance factor (φ = 0.9) in Hibbeler's design examples is a carefully chosen number derived from the normal distribution. They are the translation of statistical reliability theory into a form engineers can use with a hand calculator. When you understand the normal distribution, you understand *why* structural codes look the way they do.
